In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_PATH = "/content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp"
DATA_PATH = os.path.join(BASE_PATH, "data")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs")

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Ruta base:", BASE_PATH)
print("Ruta data:", DATA_PATH)
print("Ruta outputs:", OUTPUT_PATH)

Ruta base: /content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp
Ruta data: /content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp/data
Ruta outputs: /content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp/outputs


In [4]:
import pandas as pd

file_path = os.path.join(
    DATA_PATH,
    "musical_instruments_reviews_base.csv"
)

df_reviews = pd.read_csv(file_path)

df_reviews.head()

,reviewText,overall,summary,sentiment
0,"Not much to write about here, but it does exac...",5,good,1
1,The product does exactly as it should and is q...,5,Jake,1
2,The primary job of this device is to block the...,5,It Does The Job Well,1
3,Nice windscreen protects my MXL mic and preven...,5,GOOD WINDSCREEN FOR THE MONEY,1
4,This pop filter is great. It looks and perform...,5,No more pops when I record my vocals.,1


In [ ]:
##import os

file_path = os.path.join(DATA_PATH, "Musical_Instruments_5.json.gz")

print("Existe:", os.path.exists(file_path))
print("Tamaño en MB:", round(os.path.getsize(file_path) / (1024*1024), 2))

Existe: True
Tamaño en MB: 2.35


In [6]:
df = df_reviews
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10261 entries, 0 to 10260
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   reviewText  10254 non-null  object
 1   overall     10261 non-null  int64 
 2   summary     10261 non-null  object
 3   sentiment   10261 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 320.8+ KB


Index(['reviewText', 'overall', 'summary', 'sentiment'], dtype='object')

In [7]:
df_reviews = df[["reviewText", "overall", "summary"]].copy()

df_reviews["sentiment"] = df_reviews["overall"].apply(
    lambda x: 1 if x >= 4 else 0
)

df_reviews.head()

,reviewText,overall,summary,sentiment
0,"Not much to write about here, but it does exac...",5,good,1
1,The product does exactly as it should and is q...,5,Jake,1
2,The primary job of this device is to block the...,5,It Does The Job Well,1
3,Nice windscreen protects my MXL mic and preven...,5,GOOD WINDSCREEN FOR THE MONEY,1
4,This pop filter is great. It looks and perform...,5,No more pops when I record my vocals.,1


In [9]:
df_reviews["text"] = (
    df_reviews["summary"].fillna("") + " " +
    df_reviews["reviewText"].fillna("")
)

df_reviews[["text", "overall", "sentiment"]].head()

,text,overall,sentiment
0,"good Not much to write about here, but it does...",5,1
1,Jake The product does exactly as it should and...,5,1
2,It Does The Job Well The primary job of this d...,5,1
3,GOOD WINDSCREEN FOR THE MONEY Nice windscreen ...,5,1
4,No more pops when I record my vocals. This pop...,5,1


In [11]:
import re
import string
import nltk

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [13]:
def preprocess_text(text):
    """
    Limpia y normaliza texto para NLP
    """

    # pasar a minúsculas
    text = str(text).lower()

    # eliminar URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # eliminar números
    text = re.sub(r"\d+", "", text)

    # eliminar puntuación
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # tokenizar
    tokens = word_tokenize(text)

    clean_tokens = []

    for token in tokens:

        # eliminar stopwords y palabras cortas
        if token not in stop_words and len(token) > 2:

            # lematizar
            lemma = lemmatizer.lemmatize(token)

            clean_tokens.append(lemma)

    return " ".join(clean_tokens)

In [14]:
ejemplo = df_reviews["text"].iloc[0]

print("TEXTO ORIGINAL:")
print(ejemplo)

print("\nTEXTO PREPROCESADO:")
print(preprocess_text(ejemplo))

TEXTO ORIGINAL:
good Not much to write about here, but it does exactly what it's supposed to. filters out the pop sounds. now my recordings are much more crisp. it is one of the lowest prices pop filters on amazon so might as well buy it, they honestly work the same despite their pricing,

TEXTO PREPROCESADO:
good much write exactly supposed filter pop sound recording much crisp one lowest price pop filter amazon might well buy honestly work despite pricing


In [15]:
output_file = os.path.join(DATA_PATH, "musical_instruments_reviews_preprocessed.csv")

df_reviews.to_csv(output_file, index=False)

print("Archivo guardado en:", output_file)

Archivo guardado en: /content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp/data/musical_instruments_reviews_preprocessed.csv


## Conclusiones del preprocesamiento

En este notebook se construyó una función de preprocesamiento para normalizar las reviews antes del entrenamiento del modelo de sentimiento.

El proceso incluyó conversión a minúsculas, eliminación de URLs, números, signos de puntuación, tokenización, eliminación de stopwords y lematización.

Como resultado, se obtuvo una nueva columna llamada `text_clean`, que contiene las reviews en un formato más adecuado para aplicar modelos de bolsa de palabras y clasificación supervisada.

Este preprocesamiento reduce ruido textual y permite que los modelos se enfoquen en términos con mayor valor semántico.

In [16]:
import os

output_file = os.path.join(DATA_PATH, "musical_instruments_reviews_preprocessed.csv")

print(os.path.exists(output_file))
print(output_file)

True
/content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp/data/musical_instruments_reviews_preprocessed.csv


In [17]:
df_reviews["text"] = (
    df_reviews["summary"].fillna("") + " " +
    df_reviews["reviewText"].fillna("")
)

In [18]:
df_reviews.columns

Index(['reviewText', 'overall', 'summary', 'sentiment', 'text'], dtype='object')

In [19]:
import os

output_file = os.path.join(DATA_PATH, "musical_instruments_reviews_preprocessed.csv")

print(os.path.exists(output_file))
print(output_file)

True
/content/drive/MyDrive/KeepCoding/NLP_SEARCHING/practica_final_nlp/data/musical_instruments_reviews_preprocessed.csv


In [20]:
output_file = os.path.join(
    DATA_PATH,
    "musical_instruments_reviews_preprocessed.csv"
)

df_reviews.to_csv(output_file, index=False)

print("Archivo actualizado correctamente")

Archivo actualizado correctamente


In [22]:
df_reviews["text_clean"] = df_reviews["text"].apply(preprocess_text)

In [23]:
df_reviews.columns

Index(['reviewText', 'overall', 'summary', 'sentiment', 'text', 'text_clean'], dtype='object')

In [24]:
output_file = os.path.join(
    DATA_PATH,
    "musical_instruments_reviews_preprocessed.csv"
)

df_reviews.to_csv(output_file, index=False)

print("Archivo guardado correctamente")

Archivo guardado correctamente
